### PayPal Deep Financial Analysis
This notebook uses the supervisor multi-agent system to perform a deep financial analysis of PayPal (PYPL):
1. Create a supervisor assistant with specialized finance, research, and writing agents
2. Run a comprehensive financial analysis
3. Update the assistant for a concise executive summary
4. Revert to the original configuration

#### Setup

In [1]:
from langgraph_sdk import get_client
from dotenv import load_dotenv
import os

# ---- COLAB SETUP ----
# If running in Google Colab, set the tunnel URL here before load_dotenv().
# Make sure cloudflared tunnel is running locally: cloudflared tunnel run dev-tunnel
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in str(globals().get("__builtins__", ""))
if IN_COLAB:
    os.environ["DEPLOYMENT_URL"] = "https://dev.crossstory.com"
    os.environ["API_KEY"] = ""
    print("✅ Colab mode — using: https://dev.crossstory.com")

# ---- SETUP ----
load_dotenv()
DEPLOYMENT_URL = os.getenv("DEPLOYMENT_URL")
API_KEY = os.getenv("API_KEY")
GRAPH_ID = "supervisor_prebuilt"

print(f"🔗 Deployment URL: {DEPLOYMENT_URL}")

✅ Colab mode — using: https://dev.crossstory.com
🔗 Deployment URL: https://dev.crossstory.com


#### Connect and Create a Financial Analysis Assistant

In [2]:
# 1. Connect to the LangGraph server
client = get_client(url=DEPLOYMENT_URL, api_key=API_KEY)
print("🔗 Connected to LangGraph server")

# 2. Create a financial analysis assistant using the supervisor graph
print("🤖 Creating PayPal financial analysis assistant...")

assistant = await client.assistants.create(
    graph_id=GRAPH_ID,
    config={
        "configurable": {
            "supervisor_system_prompt": """You are an expert financial analysis director orchestrating a team of specialized agents
to produce a deep financial analysis of PayPal Holdings (PYPL).

Your workflow:
1. Route to finance_research_agent to get PayPal's latest financial data and news (ticker: PYPL)
2. Route to general_research_agent to research PayPal's business model, competition, and market position
3. Route to writing_agent to compile a comprehensive financial analysis report

Be thorough and strategic. Gather both quantitative (financial metrics) and qualitative (market position, risks) data.""",
            "supervisor_model": "openai/gpt-4.1",

            "finance_system_prompt": """You are an expert finance research analyst.
Research PayPal Holdings (ticker: PYPL) thoroughly:
- Latest earnings and revenue figures
- Key financial metrics (P/E ratio, revenue growth, active accounts, TPV)
- Recent news and analyst ratings
- Stock price performance and buyback activity
Use the finance_research tool with ticker 'PYPL' and basic_research_tool for additional context.""",
            "finance_model": "openai/gpt-4.1",
            "finance_tools": ["finance_research", "basic_research_tool", "get_todays_date"],

            "research_system_prompt": """You are an expert business and market research analyst.
Research PayPal's competitive landscape and business fundamentals:
- Business model and revenue streams (transaction fees, Braintree, Venmo, credit/BNPL)
- Key competitors (Stripe, Square/Block, Apple Pay, Google Pay, Adyen, Klarna)
- Regulatory environment and compliance challenges
- Growth opportunities: crypto, BNPL, checkout innovation, international expansion
- User base trends and merchant adoption
Use the advanced_research_tool for comprehensive research.""",
            "research_model": "openai/gpt-4.1",
            "research_tools": ["advanced_research_tool", "get_todays_date"],

            "writing_system_prompt": """You are an expert financial analyst and report writer.
Compile all research into a structured deep financial analysis report with these sections:

# PayPal Holdings (PYPL) — Deep Financial Analysis

## 1. Company Overview
## 2. Financial Performance
   - Revenue & profitability trends
   - Key metrics: TPV, active accounts, transactions per account
## 3. Business Model Analysis
   - Revenue streams breakdown (Braintree, Venmo, BNPL, credit)
   - Take rate and unit economics
## 4. Competitive Position
   - Market share and differentiation
   - Key competitors and threats
## 5. Risk Factors
   - Regulatory, competitive, and operational risks
## 6. Growth Outlook
   - Opportunities: checkout innovation, crypto, international, BNPL
## 7. Investment Summary
   - Bull case / Bear case
   - Key takeaways

Be data-driven, specific, and professional.""",
            "writing_model": "openai/gpt-4.1",
            "writing_tools": ["get_todays_date"],
        }
    },
    name="PayPal Financial Analyst"
)

print("✅ Assistant created successfully!")
print(f"   📍 Assistant ID: {assistant['assistant_id']}")
print(f"   📝 Name: {assistant['name']}")
print(f"   🔢 Version: {assistant['version']}")

🔗 Connected to LangGraph server
🤖 Creating PayPal financial analysis assistant...
✅ Assistant created successfully!
   📍 Assistant ID: 6637d5c6-b877-4e76-a35a-cccfcfbd4c96
   📝 Name: PayPal Financial Analyst
   🔢 Version: 1


In [3]:
import json

def print_event(event_data, seen_ids):
    for node_name, node_data in event_data.items():
        if not isinstance(node_data, dict):
            continue
        for msg in node_data.get("messages", []):
            msg_id = msg.get("id", "")
            if msg_id and msg_id in seen_ids:
                continue
            if msg_id:
                seen_ids.add(msg_id)

            msg_type = msg.get("type")
            msg_name = msg.get("name") or node_name
            msg_content = msg.get("content", "")
            tool_calls = msg.get("tool_calls") or msg.get("additional_kwargs", {}).get("tool_calls")

            if msg_type == "ai":
                if tool_calls:
                    for tc in (tool_calls if isinstance(tool_calls, list) else []):
                        tc_name = tc.get("name") if isinstance(tc, dict) else tc.function.name
                        print(f"🔧 [{msg_name}] → {tc_name}")
                elif msg_content and str(msg_content).strip():
                    print(f"\n💬 [{msg_name}]:\n{msg_content}\n")

            elif msg_type == "tool":
                tool_name = msg.get("name", "tool")
                content = msg.get("content", "")
                if tool_name.startswith("transfer"):
                    continue
                try:
                    results = json.loads(content)
                    print(f"✅ Tool '{tool_name}' returned {len(results) if isinstance(results, list) else 1} results")
                except:
                    print(f"✅ Tool '{tool_name}' completed")

# 3. Run the deep financial analysis
thread = await client.threads.create()
print(f"🧵 Thread: {thread['thread_id']}")
print("📊 Running deep financial analysis of PayPal (PYPL)...")
print("="*60)

seen_ids = set()
async for event in client.runs.stream(
    thread["thread_id"],
    assistant["assistant_id"],
    input={"messages": [{"role": "human", "content":
        "Perform a deep financial analysis of PayPal Holdings (PYPL). "
        "Research their financials, business model, competitive position, risks, and growth outlook. "
        "Compile everything into a comprehensive investment analysis report."}]},
    stream_mode="updates",
):
    if event.event == "metadata":
        print(f"📋 Run ID: {event.data.get('run_id', '')[:8]}...\n")
    elif event.event == "updates":
        print_event(event.data, seen_ids)

print("\n" + "="*60)
print("🎉 Full analysis complete!")
print("="*60)

🧵 Thread: 019df6ba-e185-7c60-9c3f-4e6dacccf66a
📊 Running deep financial analysis of PayPal (PYPL)...
📋 Run ID: 019df6ba...

🔧 [supervisor] → transfer_to_finance_research_agent

💬 [finance_research_agent]:
Below is a comprehensive investment analysis report for PayPal Holdings (PYPL), integrating financials, business model review, competition, risks, and growth outlook.

---

# PayPal Holdings (PYPL): Investment Analysis Report

## 1. Financial Overview

### Latest Earnings and Revenue
- Annual Payment Volume (TPV): Over $1.6 trillion (2024)
- Active Accounts: Over 420 million
- Revenue: Most recent reported quarterly revenue was $7.69 billion (Q1 2024)
- Recent Earnings: Adjusted EPS in Q1 2024 was $1.08/share

### Key Financial Metrics
- P/E Ratio: ~15 as of mid-2024 (well below sector average)
- Revenue Growth: Recent flat revenue trend, though previously growing double-digits (2021–2022)
- Total Payment Volume (TPV): Growth has slowed but outpaces some legacy competitors
- Active Ac

In [4]:
# 4. Update the assistant to produce a concise executive summary
print("🔄 Updating assistant for executive summary format...")

updated_assistant = await client.assistants.update(
    assistant["assistant_id"],
    config={
        "configurable": {
            "supervisor_system_prompt": """You are a financial analysis director.
Produce a concise executive summary of PayPal (PYPL) — max 500 words.
Route to finance_research_agent first, then directly to writing_agent.""",
            "supervisor_model": "openai/gpt-4.1",
            "writing_system_prompt": """You are a financial analyst. Write a concise executive summary of PayPal (PYPL) in max 500 words covering:
- Current financial snapshot (revenue, TPV, active accounts)
- Key strengths and risks
- Investment verdict (Buy/Hold/Sell with brief rationale)""",
            "writing_model": "openai/gpt-4.1",
            "writing_tools": ["get_todays_date"],
            "finance_tools": ["finance_research", "get_todays_date"],
            "research_tools": ["advanced_research_tool", "get_todays_date"],
        }
    },
)

print("✅ Assistant updated to executive summary mode!")
print(f"   🔢 New Version: {updated_assistant['version']}")

🔄 Updating assistant for executive summary format...
✅ Assistant updated to executive summary mode!
   🔢 New Version: 2


In [5]:
# 5. Run the executive summary version
thread2 = await client.threads.create()
print(f"🧵 Thread: {thread2['thread_id']}")
print("📝 Running executive summary analysis...")
print("="*60)

seen_ids2 = set()
async for event in client.runs.stream(
    thread2["thread_id"],
    updated_assistant["assistant_id"],
    input={"messages": [{"role": "user", "content":
        "Give me a concise executive summary and investment verdict on PayPal (PYPL)."}]},
    stream_mode="updates"
):
    if event.event == "updates":
        print_event(event.data, seen_ids2)

print("\n" + "="*60)
print("🎉 Executive summary complete!")
print("="*60)

🧵 Thread: 019df6bd-624a-7240-b338-be4ab96f209e
📝 Running executive summary analysis...
🔧 [supervisor] → transfer_to_finance_research_agent

💬 [finance_research_agent]:
**Executive Summary:**  
PayPal Holdings (PYPL) is currently trading at a notably low valuation after a steep decline in its share price. The company is making strategic efforts to streamline operations and refocus on growth initiatives. Anticipation is high around upcoming catalysts, notably its Q1 earnings report. Investor sentiment hinges on PayPal's ability to revive transaction growth and demonstrate tangible competitive advantages in the evolving digital payments landscape.

**Investment Verdict:**  
PayPal’s low valuation may present an attractive entry point for value-oriented investors, especially if the Q1 earnings show signs of financial and operational momentum. However, until PayPal proves it can reignite growth, caution is warranted. Investors should closely monitor near-term results and management’s guidan

In [ ]:
# 6. Revert to the full deep analysis version
print("⏪ Reverting to full deep analysis (Version 1)...")
await client.assistants.set_latest(assistant['assistant_id'], 1)
print("✅ Reverted successfully!")
print("   🔢 Now using: Version 1 (Full deep financial analysis)")